# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Dataset identifier: {md.identifier}")
print(f"Version: {md.version}")
print(f"License: {md.license}")
print(f"Date published: {md.datePublished}")


## 2. Data Overview
Review available record sets and their fields. All references use `@id` fields as defined in the Croissant metadata.


In [ ]:
# Collect all record sets
# Each record set contains its fields (columns) with `@id` identifiers.
record_sets = [rs for rs in dataset.metadata.recordSets]
print(f"Number of record sets: {len(record_sets)}")
pprint.pprint([
    {'@id': rs.id, 'name': getattr(rs, 'name', None), 'description': getattr(rs, 'description', None)}
    for rs in record_sets
])

# List fields (columns) for each record set by their @id
for rs in record_sets:
    print(f"\nRecord Set: {rs.id}")
    # Fields can be accessed via rs.fields
    for field in rs.fields:
        print(f"  Field: {field.id}, Name: {getattr(field, 'name', None)}, DataType: {getattr(field, 'dataType', None)}")

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Select the primary record set, typically the one containing measurements/observations
# Use the @id from above. Here, we choose the first (and likely primary, as this is a single table, mass-spectrometry dataset).
if len(record_sets) == 0:
    raise RuntimeError('No record sets found in the dataset metadata.')

main_record_set_id = record_sets[0].id
print(f"Main record set @id: {main_record_set_id}")

# List of record set @ids for possible extension/iteration
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    # Each element produced by .records() is a dict with field @id as keys
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Preview columns and top rows
df = dataframes[main_record_set_id]
print("Columns in main record set:")
pp = pprint.PrettyPrinter()
pp.pprint(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on a numeric field (e.g., normalized abundance or peptide count), normalizing values, and grouping by a categorical field (e.g., experimental condition). All references use field `@id`s from the metadata.

In [ ]:
# Get numeric and group field candidates by their @id
main_rs = record_sets[0]
fields = main_rs.fields

# Locate candidate numeric fields
numeric_fields = [field.id for field in fields if getattr(field, 'dataType', None) in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number']]
group_fields = [field.id for field in fields if getattr(field, 'dataType', None) == 'Text' or 'name' in getattr(field, 'id', '')]

print('Available numeric fields:', numeric_fields)
print('Possible group fields:', group_fields)

# For illustrative purposes, pick first numeric field
if len(numeric_fields) == 0:
    raise ValueError('No numeric fields found for EDA!')
numeric_field = numeric_fields[0]
print(f'Using numeric field: {numeric_field}')

# Filter values higher than a threshold (e.g., 10)
threshold = 10
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f'{numeric_field}_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) \
        / filtered_df[numeric_field].astype(float).std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Pick a group field if available
    group_field = None
    if len(group_fields) > 0:
        group_field = group_fields[0]
        print(f'Grouping by: {group_field}')
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data (showing averages) by field '{group_field}':")
            print(grouped_df.head())
    else:
        print('No categorical group field identified for grouping.')
else:
    print(f"Numeric field '{numeric_field}' not found among DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. You can alter the fields below to plot different aspects as desired.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot of the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].astype(float).dropna(), bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouped by a field, plot normalized field group means (if available)
if 'grouped_df' in locals() and grouped_df.shape[0] < 30:
    grouped_df[norm_col].plot(kind='bar')
    plt.ylabel(norm_col)
    plt.xlabel(group_field)
    plt.title(f'Normalized {numeric_field} by {group_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded Croissant-structured dataset metadata and data using `mlcroissant`, previewed the available record sets and fields using their `@id`, loaded primary records into a pandas DataFrame, filtered and normalized a numeric field, optionally grouped data, and visualized data distributions. This workflow can be adapted to any Croissant dataset by referencing entity `@id`s for robust and reproducible FAIR^2 data analyses.